In [ ]:
import os
print(f"Prefect server URL used internally: {os.environ['PREFECT_API_URL']}")
dashboard = f"{os.environ['RSPY_PREFECT_URL']}/dashboard"
print(f"Prefect dashboard public URL: {dashboard}")

In [ ]:
# Init environment before running a demo notebook.
from resources.utils import *  
from resources.dask_utils import *

init_demo()

# Reload the global vars again
from resources.utils import *  
from resources.dask_utils import *  
# Other imports
import getpass
import os
from importlib import reload

from rs_common import prefect_utils
from rs_common.prefect_utils import *

# Get the prefect share bucket folder
share_bucket = await get_share_bucket()

In [ ]:
# Use a subfolder named after the current user
s3_code_folder = f"users/{OWNER_ID}/code" 

if local_mode:
    print (f"S3 MinIO dashboard: http://localhost:9101 with user=minio password=Strong#Pass#1234")
print(f"Upload local source code to: 's3://{share_bucket.bucket_name}/{share_bucket.bucket_folder}/{s3_code_folder}'")

# Upload local directory and resources contents
await share_bucket.put_directory(local_path = ".", to_path = s3_code_folder)
await share_bucket.put_directory(local_path = "../../resources", to_path = f"{s3_code_folder}/resources")

# Pass the full S3 code folder as an environment variable
os.environ["S3_CODE_FOLDER"] = f"{share_bucket.bucket_folder}/{s3_code_folder}"

In [ ]:
%%bash
# Deploy the flow. We don't need to be in the git root folder.
prefect --no-prompt deploy --prefect-file "./lazy_flow_deployment.yaml"
prefect --no-prompt deploy --prefect-file "./main_flow_deployment.yaml"

In [ ]:
deploy_name = "lazy-flow-deployment/lazy_flow_deployment"
await prefect_utils.wait_for_deployment(deploy_name)
deploy_name = "main-flow-deployment/deployment_flow"
await prefect_utils.wait_for_deployment(deploy_name)

In [ ]:
%%bash -s "main-flow-deployment/deployment_flow"
# Trigger a run for this flow from the command line
prefect deployment run "$1" --watch